In [5]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv(r"spam.csv", encoding='latin-1')

# Preprocess
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['message'])
y = df['label']

# Total features
total_features = X.shape[1]
print("Total number of features:", total_features)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model (NO penalty parameter)
model = LogisticRegression(
    solver='saga',
    l1_ratio=1,     # L1 effect
    max_iter=1000
)

model.fit(X_train, y_train)

coef = model.coef_[0]

non_zero = np.sum(coef != 0)
zero = np.sum(coef == 0)

print("\nFor default regularization:")
print("Non-zero features:", non_zero)
print("Eliminated features:", zero)

# Different C values
print("\nFeature selection with different C values:")
for c in [0.01, 0.1, 1]:
    model = LogisticRegression(
        solver='saga',
        l1_ratio=1,
        C=c,
        max_iter=1000
    )
    
    model.fit(X_train, y_train)
    selected = np.sum(model.coef_[0] != 0)
    
    print(f"C = {c} --> Selected Features: {selected}")

# Reduction %
reduction = ((total_features - non_zero) / total_features) * 100
print("\nPercentage reduction in features:", reduction, "%")

Total number of features: 8404

For default regularization:
Non-zero features: 100
Eliminated features: 8304

Feature selection with different C values:
C = 0.01 --> Selected Features: 0
C = 0.1 --> Selected Features: 7
C = 1 --> Selected Features: 100

Percentage reduction in features: 98.81009043312709 %
